In [1]:
# ============================================================
# FASE 5 — PREPROCESSING & CLEANSING
# Cell 1: Setup + Fix DFT-02 (Vibration Negatif) + DFT-01 Info
# ============================================================

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# [1] SYS.PATH SETUP
# Notebook: notebooks/fase_5_preprocessing/
# ML_ROOT  = dua level ke atas
# ----------------------------------------------------------
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# ----------------------------------------------------------
# [2] IMPORT CONFIG
# ----------------------------------------------------------
from config import DATA_INTERIM_DIR, DATA_PROCESSED_DIR, GLOBAL_SEED

np.random.seed(GLOBAL_SEED)

# ----------------------------------------------------------
# [3] LOAD DATA
# ----------------------------------------------------------
INPUT_PATH = DATA_INTERIM_DIR / "df_sensor_featured.parquet"

df = pd.read_parquet(INPUT_PATH)

df = (
    df
    .sort_values(["machine_id", "timestamp"], ascending=True)
    .reset_index(drop=True)
)

# ----------------------------------------------------------
# [4] KONFIRMASI SETUP
# ----------------------------------------------------------
SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  SETUP KONFIRMASI — FASE 5 PREPROCESSING")
print(SEP)
print(f"\n  SRC_PATH           : {SRC_PATH}")
print(f"  GLOBAL_SEED        : {GLOBAL_SEED}")
print(f"  INPUT_PATH         : {INPUT_PATH.name}")
print(f"  DATA_INTERIM_DIR   : {DATA_INTERIM_DIR}")
print(f"  DATA_PROCESSED_DIR : {DATA_PROCESSED_DIR}")
print(f"\n  Shape df           : {df.shape}")
print(f"  Jumlah kolom       : {df.shape[1]}")
print(f"\n  timestamp min      : {df['timestamp'].min()}")
print(f"  timestamp max      : {df['timestamp'].max()}")
print(SEP)

# ════════════════════════════════════════════════════════════
# FIX DFT-02 — VIBRATION NEGATIF
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  FIX DFT-02 — VIBRATION NEGATIF")
print(SEP)

AFFECTED_COLS = [
    "vibration",
    "vibration_roll_mean_24h", "vibration_roll_mean_48h",
    "vibration_roll_max_24h",  "vibration_roll_max_48h",
    "vibration_lag_6h", "vibration_lag_12h", "vibration_lag_24h",
    "temp_per_vibration", "noise_per_vibration",
]

RATIO_COLS = ["temp_per_vibration", "noise_per_vibration"]
CLIP_COLS  = [c for c in AFFECTED_COLS if c not in RATIO_COLS]

# Defensive: hanya proses kolom yang ada di df
affected_ok = [c for c in AFFECTED_COLS if c in df.columns]
clip_ok     = [c for c in CLIP_COLS     if c in df.columns]
ratio_ok    = [c for c in RATIO_COLS    if c in df.columns]

# ----------------------------------------------------------
# [A] SEBELUM FIX
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [A] NILAI <= 0 SEBELUM FIX")
print(sep)
print()

for col in affected_ok:
    n = int((df[col] <= 0).sum())
    flag = " ← akan difix" if n > 0 else ""
    print(f"    {col:<40} : {n:>8,}{flag}")

# ----------------------------------------------------------
# [B] FIX: CLIP kolom sensor asli, rolling, lag
# ----------------------------------------------------------
for col in clip_ok:
    df[col] = df[col].clip(lower=0)

# ----------------------------------------------------------
# [C] FIX: HITUNG ULANG kolom ratio dari nilai bersih
# ----------------------------------------------------------
df["temp_per_vibration"]  = df["temperature"] / (df["vibration"] + 1e-9)
df["noise_per_vibration"] = df["noise_level"] / (df["vibration"] + 1e-9)

for col in ratio_ok:
    n_inf = int(np.isinf(df[col]).sum())
    if n_inf > 0:
        df[col]    = df[col].replace([np.inf, -np.inf], np.nan)
        col_median = df[col].median()
        df[col]    = df[col].fillna(col_median)
        print(f"\n  [INFO] {col}: {n_inf:,} Inf diganti median ({col_median:.6f})")

# ----------------------------------------------------------
# [D] SETELAH FIX: verifikasi nilai <= 0 (harus semua 0)
# ----------------------------------------------------------
print(f"\n{sep}")
print("  [B] NILAI <= 0 SETELAH FIX (harus semua 0)")
print(sep)
print()

all_clean = True
for col in affected_ok:
    n    = int((df[col] <= 0).sum())
    flag = "[OK]  " if n == 0 else "[WARN]"
    print(f"    {flag}  {col:<40} : {n:,}")
    if n > 0:
        all_clean = False

print()
if all_clean:
    print("  [OK] DFT-02 CLOSED — semua nilai vibration & turunannya >= 0.")
else:
    print("  [WARN] DFT-02 belum sepenuhnya resolved.")

# ════════════════════════════════════════════════════════════
# INFO DFT-01 — PARTS_REPLACED
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  INFO DFT-01 — PARTS_REPLACED")
print(SEP)

if "parts_replaced" not in df.columns:
    print("\n  [OK] DFT-01 CLOSED.")
    print("  Kolom 'parts_replaced' TIDAK ada di df sensor (tidak di-merge).")
    print("  Tidak ada dampak ke pipeline modeling.")
else:
    n_nan = int(df["parts_replaced"].isna().sum())
    print(f"\n  [INFO] Kolom 'parts_replaced' ada di df ({n_nan:,} NaN).")
    print("  Penanganan: fill → 'Unknown' di cell selanjutnya.")

# ════════════════════════════════════════════════════════════
# VALIDASI AKHIR
# ════════════════════════════════════════════════════════════

print(f"\n{SEP}")
print("  VALIDASI AKHIR CELL 1")
print(SEP)

total_nan = int(df.isna().sum().sum())
num_cols  = df.select_dtypes(include=[np.number]).columns
total_inf = int(np.isinf(df[num_cols].values).sum())
shape_ok  = df.shape == (100_000, 75)

print(f"\n  Total NaN di seluruh df     : {total_nan:,}  "
      f"→  {'[OK]' if total_nan == 0 else '[WARN]'}")
print(f"  Total Inf di kolom numerik  : {total_inf:,}  "
      f"→  {'[OK]' if total_inf == 0 else '[WARN]'}")
print(f"  Shape df                    : {df.shape}  "
      f"→  {'[OK]' if shape_ok else '[WARN] expected (100000, 75)'}")

print(f"\n{SEP}")
print("  [OK] Setup & Defect Fix selesai. df siap untuk Cell 2.")
print(SEP)


  SETUP KONFIRMASI — FASE 5 PREPROCESSING

  SRC_PATH           : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\src
  GLOBAL_SEED        : 42
  INPUT_PATH         : df_sensor_featured.parquet
  DATA_INTERIM_DIR   : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\interim
  DATA_PROCESSED_DIR : C:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\data\processed

  Shape df           : (100000, 75)
  Jumlah kolom       : 75

  timestamp min      : 2025-07-01 00:00:00
  timestamp max      : 2026-01-25 07:00:00

  FIX DFT-02 — VIBRATION NEGATIF

-----------------------------------------------------------------
  [A] NILAI <= 0 SEBELUM FIX
-----------------------------------------------------------------

    vibration                                :        2 ← akan difix
    vibration_roll_mean_24h                

In [2]:
# ============================================================
# FASE 5 — Cell Re-Validasi DFT-02
# Cek nilai STRICTLY NEGATIF (< 0), bukan <= 0
# Nilai 0 adalah VALID (mesin idle/berhenti)
# ============================================================

SEP = "=" * 65
sep = "-" * 65

AFFECTED_COLS = [
    "vibration",
    "vibration_roll_mean_24h", "vibration_roll_mean_48h",
    "vibration_roll_max_24h",  "vibration_roll_max_48h",
    "vibration_lag_6h", "vibration_lag_12h", "vibration_lag_24h",
    "temp_per_vibration", "noise_per_vibration",
]

# Defensive: hanya kolom yang ada di df
affected_ok = [c for c in AFFECTED_COLS if c in df.columns]

print(SEP)
print("  RE-VALIDASI DFT-02 — STRICTLY NEGATIF (< 0)")
print(SEP)
print("  Catatan: nilai 0 adalah VALID (mesin idle/berhenti)")
print(f"  Kolom dicek : {len(affected_ok)}")
print()
print(f"  {'Kolom':<40}  {'Negatif (<0)':>12}  {'Zero (=0)':>10}  Status")
print(f"  {'-'*75}")

total_negative = 0

for col in affected_ok:
    n_neg  = int((df[col] < 0).sum())
    n_zero = int((df[col] == 0).sum())
    status = "✅ OK" if n_neg == 0 else "🔴 MASIH ADA NEGATIF"
    print(f"  {col:<40}  {n_neg:>12,}  {n_zero:>10,}  {status}")
    total_negative += n_neg

print(f"\n{sep}")
print("  KESIMPULAN")
print(sep)

if total_negative == 0:
    print("\n  ✅ DFT-02 FULLY RESOLVED")
    print("  Tidak ada satupun nilai negatif pada kolom vibration & turunannya.")
    print("  Nilai zero (=0) adalah VALID — merepresentasikan mesin idle/berhenti.")
    print("  Clip ke 0 bekerja dengan benar. DFT-02 dinyatakan CLOSED.")
else:
    print(f"\n  🔴 MASIH ADA {total_negative:,} NILAI NEGATIF — perlu investigasi lanjutan.")
    print("  Periksa kolom yang ditandai di atas.")

print(f"\n{sep}")
print("  CEK SHAPE FINAL")
print(sep)
shape_ok = df.shape == (100_000, 75)
flag     = "[OK]" if shape_ok else "[WARN]"
print(f"\n  Shape df : {df.shape}  →  {flag}")
if not shape_ok:
    print("  Expected : (100000, 75)")

print(f"\n{SEP}")
print("  [OK] Re-Validasi DFT-02 selesai.")
print(SEP)


  RE-VALIDASI DFT-02 — STRICTLY NEGATIF (< 0)
  Catatan: nilai 0 adalah VALID (mesin idle/berhenti)
  Kolom dicek : 10

  Kolom                                     Negatif (<0)   Zero (=0)  Status
  ---------------------------------------------------------------------------
  vibration                                            0           2  ✅ OK
  vibration_roll_mean_24h                              0           0  ✅ OK
  vibration_roll_mean_48h                              0           0  ✅ OK
  vibration_roll_max_24h                               0           0  ✅ OK
  vibration_roll_max_48h                               0           0  ✅ OK
  vibration_lag_6h                                     0           2  ✅ OK
  vibration_lag_12h                                    0           2  ✅ OK
  vibration_lag_24h                                    0           2  ✅ OK
  temp_per_vibration                                   0           0  ✅ OK
  noise_per_vibration                             